# 02 · Ingeniería de características y preprocesamiento
**Flujo:** dataset limpio → eliminar identificadores → split estratificado → (solo con train) elegir imputador, validar variables nuevas y estrategia de balanceo → Pipeline → `datos/model/`.

**Regla de oro:** el split ocurre **antes** de cualquier transformación que aprenda parámetros.

In [ ]:
# === Solo Google Colab (en Jupyter local esta celda no hace nada) ===
import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    # Ajusta esta ruta si subiste la carpeta a otro lugar de tu Drive
    RUTA_PROYECTO = "/content/drive/MyDrive/proyecto-clasificacion-musical"
    os.chdir(RUTA_PROYECTO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    # os.environ["MODO_RAPIDO"] = "1"   # descomenta para una corrida rápida (~10 min en vez de ~85)

In [1]:
# === Configuración común ===
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "funciones.py").exists())
sys.path.insert(0, str(RAIZ / "src"))
import funciones as F

warnings.filterwarnings("ignore", message=".*Early stopping criterion not reached.*")  # IterativeImputer
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)
rutas = F.obtener_rutas(RAIZ)
# MODO_RAPIDO=1 (variable de entorno) reduce muestras y presupuestos para iterar rápido.
MODO_RAPIDO = os.environ.get("MODO_RAPIDO", "0") == "1"
print("Raíz:", rutas.raiz, "| modo rápido:", MODO_RAPIDO)

Raíz: <raíz> | modo rápido: False


In [2]:
tipos = {c: str for c in F.COLUMNAS_CATEGORICAS + [F.COLUMNA_ARTISTA]}
df = pd.read_csv(rutas.processed / "music_genre_limpio.csv", dtype=tipos)
df.shape

(46198, 18)

## 1. Eliminación de variables
* `instance_id`: clave técnica. `obtained_date`: fecha del scraping (4 valores), no es una propiedad musical. `track_name`: casi única por fila; como categoría solo memoriza.
* `artist_name` (alta cardinalidad) **se conserva** y se codifica. Antes conviene ver cuánta información aporta:

In [3]:
por_artista = df.dropna(subset=[F.COLUMNA_ARTISTA]).groupby(F.COLUMNA_ARTISTA)[F.OBJETIVO]
print("Artistas únicos:", por_artista.ngroups)
print("Canciones por artista (mediana):", por_artista.size().median())
por_artista.nunique().value_counts(normalize=True).rename("prop. artistas por nº de géneros").to_frame()

Artistas únicos: 6862
Canciones por artista (mediana): 2.0


,prop. artistas por nº de géneros
music_genre,
1,0.8328
2,0.1470
3,0.0195
4,0.0006


~80 % de los artistas tiene **un solo género**: el artista es casi una etiqueta. Opciones:
| Opción | Ventaja | Riesgo |
|---|---|---|
| OneHot | exacto | 6 800+ columnas dispersas; inviable para MLP/NB |
| **Target Encoding** (elegido) | codifica P(género \| artista) en K columnas con *shrinkage* hacia la prior | fuga de `y` si se ajusta y aplica sobre las mismas filas → se evita con el **cross-fitting interno** de `sklearn.TargetEncoder` |
| Frequency Encoding | no usa `y` (sin riesgo de fuga) | solo mide la popularidad del artista en el catálogo |
| Excluir | mide generalización a artistas nuevos | renuncia a la variable más informativa |

El cuadernillo 03 cuantifica cuánto del rendimiento depende del artista (validación agrupada por artista).

In [4]:
X, y = F.separar_X_y(df)
X_train, X_test, y_train, y_test = F.dividir_train_test(X, y, test_size=0.2)
pd.DataFrame({"train %": y_train.value_counts(normalize=True) * 100,
              "test %": y_test.value_counts(normalize=True) * 100}).round(2)

,train %,test %
music_genre,,
Anime,10.7900,10.7800
Classical,10.7400,10.7500
Electronic,10.6100,10.6200
Blues,10.4600,10.4500
Jazz,10.4400,10.4400
Country,10.3100,10.3100
Alternative,9.5500,9.5600
Rock,9.4500,9.4500
Hip-Hop,8.9100,8.9100


## 2. Variables con sentido musical (validadas solo con train)
`F.IngenieriaMusical` es un transformador **sin estado** que se ubica *después* de la imputación en el Pipeline. Aquí se evalúa su poder discriminante en train (η² de Kruskal-Wallis) frente a las variables originales.

In [5]:
ing = F.IngenieriaMusical(eliminar_originales=False).fit(X_train[F.COLUMNAS_NUMERICAS])
train_fe = ing.transform(X_train[F.COLUMNAS_NUMERICAS]).assign(**{F.OBJETIVO: y_train.values})
nuevas = ["log_duration", "energia_acustica", "pulso_bailable", "intensidad", "vocalidad",
          "log_speechiness", "log_liveness", "log_instrumental"]
F.kruskal_por_genero(train_fe, nuevas + ["duration_ms", "energy", "acousticness", "tempo", "danceability"])

,H,p_valor,eta2_H,magnitud
variable,,,,
log_instrumental,14371.6000,0.0000,0.3887,grande
danceability,13053.8000,0.0000,0.3531,grande
vocalidad,12276.4000,0.0000,0.3320,grande
energia_acustica,11132.6000,0.0000,0.3011,grande
acousticness,10916.9000,0.0000,0.2952,grande
intensidad,10352.2000,0.0000,0.2799,grande
pulso_bailable,9247.1000,0.0000,0.2777,grande
energy,10107.2000,0.0000,0.2733,grande
log_speechiness,9770.8000,0.0000,0.2642,grande


**Lectura:**
* `vocalidad` (η² ≈ 0.33) supera a `speechiness` (≈ 0.26) y `energia_acustica` (≈ 0.30) supera a `energy` y `acousticness` por separado: las combinaciones resumen mejor el eje musical que sus componentes.
* `pulso_bailable` (≈ 0.28) **no** supera a `danceability` (≈ 0.35) por sí sola, pero aporta la interacción tempo × ritmo que un modelo lineal no puede construir. Se conserva y se deja que la regularización decida.
* `log_duration` tiene **exactamente** el mismo η² que `duration_ms`: Kruskal-Wallis usa rangos y el log es monótono. Lo mismo ocurre con los árboles. El log se justifica para los modelos que usan **distancias o gradientes** (LogReg, MLP, GaussianNB, SMOTE), donde la cola de piezas de más de 60 minutos deformaría la escala.

Se guarda también una versión del dataset con estas variables (sin imputar, con NaN donde falta el dato original) en `datos/processed/` para consultas de negocio.

In [6]:
procesado = df.drop(columns=F.COLUMNAS_ID).join(
    F.IngenieriaMusical(eliminar_originales=False).fit_transform(df[F.COLUMNAS_NUMERICAS])[nuevas])
procesado.to_csv(rutas.processed / "music_genre_procesado.csv", index=False)
procesado.shape

(46198, 23)

## 3. Elección del imputador (experimento de enmascaramiento sobre train)
Se ocultan el 10 % de los valores conocidos de `tempo` y `duration_ms` y se mide el error de reconstrucción. NRMSE = 1 equivale a imputar la media.

In [7]:
F.evaluar_imputadores(X_train, muestra=5000 if MODO_RAPIDO else 12000)

NRMSE               RMSE            segundos  \
variable                  duration_ms  tempo duration_ms   tempo duration_ms   
imputador                                                                      
Iterative (BayesianRidge)      0.9860 0.9535 106149.7200 29.3500      0.5100   
KNN (k=15)                     0.9654 0.9450 103929.1000 29.0900      3.7000   
Mediana global                 1.0261 1.0188 110466.0300 31.3600      0.0400   

                                  
variable                   tempo  
imputador                         
Iterative (BayesianRidge) 0.5100  
KNN (k=15)                3.7000  
Mediana global            0.0400

**Decisión:** ninguna variable se reconstruye bien (NRMSE ≈ 0.9–1): el tempo y la duración son poco predecibles a partir de las demás. KNN gana por poco en `duration_ms` e Iterative en `tempo`, ambos con mejoras de solo 5–8 % frente a la mediana. Se elige **IterativeImputer** porque en producción es más barato (KNN tiene que guardar todo train y medir distancias contra él para cada canción nueva) y porque con una diferencia tan pequeña el costo pesa más. Se añade `add_indicator=True` para que el modelo decida si el hecho de que falte el dato es informativo. `F.construir_preprocesador(imputador="knn")` permite cambiarlo.

## 4. Pipeline de preprocesamiento
```
ColumnTransformer
├── num  : IterativeImputer(+indicador) → IngenieriaMusical → StandardScaler
├── cat  : moda → OneHotEncoder(key, mode)
└── artista: '__desconocido__' → TargetEncoder(multiclase, cross-fitting)
→ [SMOTE solo en fit] → clasificador
```
**StandardScaler vs RobustScaler:** después del log, las variables quedan casi simétricas y StandardScaler deja todo en unidades de desviación estándar (necesario para LogReg, MLP y SMOTE, que usan distancias o gradientes). RobustScaler (mediana/IQR) queda como opción por si se omiten las transformaciones log.

In [8]:
from sklearn.linear_model import LogisticRegression
pipe = F.construir_pipeline(LogisticRegression(max_iter=3000))
display(pipe)
Z_train = F.nombres_salida(pipe, X_train, y_train)
print("Matriz transformada:", Z_train.shape)
Z_train.describe().T[["mean", "std"]].head(20)

,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The g

Matriz transformada: (36958, 41)


,mean,std
num__popularity,0.0000,1.0000
num__acousticness,-0.0000,1.0000
num__danceability,-0.0000,1.0000
num__energy,-0.0000,1.0000
num__loudness,-0.0000,1.0000
num__tempo,-0.0000,1.0000
num__valence,-0.0000,1.0000
num__missingindicator_duration_ms,-0.0000,1.0000
num__missingindicator_tempo,-0.0000,1.0000
num__log_duration,-0.0000,1.0000


**Comprobación anti-fuga:** el preprocesador se ajustó **solo con train**. Al transformar test se usan las medias, los imputadores y las codificaciones de train, así que las medias de test no son exactamente 0, que es lo esperado y sano:

In [9]:
prep = pipe.named_steps["prep"].fit(X_train, y_train)
Z_test = prep.transform(X_test)
pd.DataFrame({"media train": Z_train.mean().head(8), "media test": Z_test.mean().head(8)})

,media train,media test
num__popularity,0.0000,-0.0120
num__acousticness,-0.0000,-0.0073
num__danceability,-0.0000,-0.0042
num__energy,-0.0000,-0.0009
num__loudness,-0.0000,-0.0033
num__tempo,-0.0000,0.0071
num__valence,-0.0000,-0.0102
num__missingindicator_duration_ms,-0.0000,0.0018


## 5. Desbalanceo: diagnóstico y estrategia
Las 10 clases empiezan perfectamente balanceadas, pero la limpieza (deduplicación) y las decisiones de negocio cambian eso. Se comparan 4 escenarios:

In [10]:
y_urbano = F.agrupar_subgeneros(y_train)  # Rap + Hip-Hop -> Urbano
X_sim, y_sim = F.simular_desbalanceo(X_train, y_train, {"Classical": 0.15, "Jazz": 0.3, "Blues": 0.5})
F.comparar_distribuciones({"train limpio": y_train, "Rap+Hip-Hop→Urbano": y_urbano,
                           "catálogo simulado": y_sim})

,train limpio,Rap+Hip-Hop→Urbano,catálogo simulado
Alternative,3530,3530,3530
Anime,3986,3986,3986
Blues,3864,3864,1932
Classical,3970,3970,596
Country,3812,3812,3812
Electronic,3922,3922,3922
Hip-Hop,3292,0,3292
Jazz,3859,3859,1158
Rap,3231,0,3231
Rock,3492,3492,3492


**Criterio de decisión** (balance de Shannon = H(y)/log K; IR = n_max/n_min):
* **IR < 1.5** (train limpio, ~1.23): desbalanceo leve. Con métricas macro y split estratificado no hace falta remuestrear; `class_weight` es un ajuste gratuito.
* **1.5 ≤ IR < ~4** (agrupación *Urbano*, IR ≈ 2): `class_weight='balanced'`. Re-pondera la pérdida con w_k = n / (K·n_k), es decir, cambia la prior sin inventar datos y no cuesta cómputo.
* **IR alto o clase minoritaria con pocas muestras** (catálogo simulado, IR ≈ 6): **SMOTE dentro del pipeline**, que interpola x_i + λ(x_vecino − x_i) **solo en los folds de entrenamiento**. Aplicarlo antes del split llenaría validación/test con vecinos sintéticos de puntos de train (fuga).

Lo comprobamos empíricamente:

In [11]:
def submuestra(X_, y_, n=6000):
    if not MODO_RAPIDO:
        return X_, y_
    i = X_.sample(min(n, len(X_)), random_state=F.SEMILLA).index
    return X_.loc[i], y_.loc[i]

lr = LogisticRegression(max_iter=3000)
res_bal = {nombre: F.comparar_balanceo(lr, *submuestra(Xe, ye), cv=F.cv_estratificada(3))
           for nombre, (Xe, ye) in {"train limpio": (X_train, y_train),
                                    "catálogo simulado": (X_sim, y_sim)}.items()}
pd.concat(res_bal)

f1_macro  balanced_accuracy  \
                  estrategia                                             
train limpio      Sin balanceo               0.8094             0.8102   
                  class_weight='balanced'    0.8088             0.8093   
                  SMOTE (en pipeline)        0.8091             0.8096   
catálogo simulado Sin balanceo               0.7896             0.7862   
                  class_weight='balanced'    0.7771             0.7913   
                  SMOTE (en pipeline)        0.7810             0.7926   

                                           log_loss  tiempo_fit_s  
                  estrategia                                       
train limpio      Sin balanceo               0.4509        1.6910  
                  class_weight='balanced'    0.4515        4.0290  
                  SMOTE (en pipeline)        0.4508        2.6922  
catálogo simulado Sin balanceo               0.5152        3.2809  
                  class_weight='balanced'    0.5319        3.6731  
                  SMOTE (en pipeline)        0.5269        5.3633

**Lectura:**
* **Train limpio (IR ≈ 1.2):** las tres estrategias empatan dentro del error de CV. **Decisión: sin remuestreo.** SMOTE solo agrega costo.
* **Catálogo simulado (IR ≈ 6):** `class_weight` y SMOTE suben la *balanced accuracy* (recuperan recall en Classical/Jazz/Blues), pero el F1-macro casi no cambia y el log-loss empeora, porque el modelo aprende una prior que no es la real. **Cuándo aplicarlos:** cuando el costo de negocio de no detectar la clase minoritaria es mayor que el de una falsa alarma. Si además se necesitan probabilidades reales, se corrige la prior con Bayes (`F.actualizacion_bayesiana`, cuadernillo 03).

## 6. Persistir conjuntos para modelado
`datos/model/` guarda los datos **limpios pero sin transformar**: el Pipeline completo se ajusta en 03 y en el entrenamiento final, para que cada fold de validación cruzada repita imputación, codificación y escalado sin fuga.

In [12]:
F.guardar_conjuntos(rutas, X_train, X_test, y_train, y_test)
sorted(p.name for p in rutas.model_data.glob("*.csv"))

['X_test.csv', 'X_train.csv', 'y_test.csv', 'y_train.csv']